# Real-Time Vehicle Detection Using YOLOv8

This notebook fine-tunes a pretrained YOLOv8s model to detect vehicles in traffic images and videos.

The workflow includes environment verification, dataset validation, model training, model evaluation, and video prediction.

## 1. Environment and GPU Verification

This section verifies the selected Python environment and checks whether PyTorch can access the NVIDIA GPU through CUDA.

In [1]:
import sys
print(sys.executable)

d:\Self Projects\Real-Time-Vehicle-Detection-and-Counting-YOLOv8-main\.venv\Scripts\python.exe


In [2]:
import torch

print('PyTorch version : ', torch.__version__)
print('Cuda : ', torch.cuda.is_available())

if torch.cuda.is_available():
    print('Cuda available',torch.cuda.get_device_name(0))
else:
    print('No cuda')

PyTorch version :  2.11.0+cu128
Cuda :  True
Cuda available NVIDIA GeForce RTX 5070 Laptop GPU


## 2. Import Required Libraries

- `os` is used to access dataset files and folders.
- `Pillow` is used to open and verify images.
- `Ultralytics YOLO` is used for model loading, training, and prediction.

In [3]:
import os
from PIL import Image
from ultralytics import YOLO

## 3. Load the Pretrained YOLOv8 Model

The pretrained YOLOv8 Small model (`yolov8s.pt`) is loaded using the Ultralytics library.

This pretrained model provides existing object-detection knowledge and will later be fine-tuned on the custom vehicle dataset.

In [4]:
print("Ultralytics imported successfully")

model = YOLO("yolov8s.pt") 

print("Model loaded successfully")


Ultralytics imported successfully
Model loaded successfully


## 4. Verify the Dataset

This section counts the training and validation images and labels.

Each image must have a corresponding YOLO annotation file. Matching image and label counts help confirm that the dataset is correctly prepared before training.

In [5]:
train_img = len(os.listdir(r'archive (1)\Vehicle_Detection_Image_Dataset\train\images'))
train_labels = len(os.listdir(r'archive (1)\Vehicle_Detection_Image_Dataset\train\labels'))

valid_img = len(os.listdir(r'archive (1)\Vehicle_Detection_Image_Dataset\valid\images'))
valid_labels = len(os.listdir(r'archive (1)\Vehicle_Detection_Image_Dataset\valid\labels'))

if train_img == train_labels and valid_img == valid_labels:
    print('Lengths are same')
else:
    print('Need correction')
    print(train_img)
    print(train_labels)
    print(valid_img)
    print(valid_labels)

Lengths are same


## 5. Create the Dataset Configuration File

The `data.yaml` file defines:

- The dataset location
- The training-image folder
- The validation-image folder
- The object class name

YOLOv8 uses this file during model training.

In [6]:
data_yaml = """
path: archive (1)\Vehicle_Detection_Image_Dataset

train: train/images
val: valid/images

names:
  0: vehicle
"""

with open(r'data.yaml', 'w') as f:
    f.write(data_yaml)

print('data.yaml created successfully')


data.yaml created successfully


## 6. Verify a Sample Training Image

This section opens one image from the training dataset and prints its dimensions.

It confirms that the image path is correct and the dataset image can be loaded successfully.

In [8]:
img_path = r'archive (1)\Vehicle_Detection_Image_Dataset\train\images\1_mp4-1_jpg.rf.9115af93de4cbf89e48a9c33dfe996c8.jpg'

img = Image.open(img_path)
print('Image size:', img.size)   # (width, height)


Image size: (640, 640)


## 7. Fine-Tune the YOLOv8s Model

The pretrained YOLOv8s model is fine-tuned on the custom vehicle dataset for 30 epochs.

GPU device `0` is used to accelerate the training process. After training, Ultralytics automatically saves the model weights and evaluation results.

In [9]:
results = model.train(
    data='data.yaml',
    epochs=30,
    imgsz=640,
    batch=8,
    workers=0,
    device=0,
    pretrained=True,
    project='runs',   
    name='train1'         
)

Ultralytics 8.4.108  Python-3.11.9 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train1, nbs=64, nms=False, op

## 8. Verify the Best Trained Model

This section stores the path of `best.pt` and checks whether the trained model file exists.

`best.pt` contains the model checkpoint with the best validation performance and is used for final prediction.

In [12]:
weights_path = r"runs/detect/runs/train1/weights/best.pt"
print('Exists:', os.path.exists(weights_path))


Exists: True


## 9. Test the Trained Model on a Video

The best trained model is loaded and tested on a sample traffic video.

A confidence threshold of `0.4` is used, and every video frame is processed using the GPU. The annotated prediction video is saved automatically inside the `runs` folder.

In [16]:
model = YOLO(weights_path)

results = model.predict(
    source= r'archive (1)\Vehicle_Detection_Image_Dataset\sample_video.mp4',            # path to your input video
    save=True,                       # save output video
    project="runs",                   # where to save
    name="video_test1",              # folder name for this test
    conf=0.4,                        # confidence threshold (0–1)
    vid_stride=1,                     # use every frame
    device=0
)

print('Done... Check the output folder.')



WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/601) d:\Self Projects\Real-Time-Vehicle-Detection-and-Counting-YOLOv8-main\archive (1)\Vehicle_Detection_Image_Dataset\sample_video.mp4: 384x640 3 vehicles, 24.3ms
video 1/1 (frame 2/601) d:\Self Projects\Real-Time-Vehicle-Detection-and-Counting-YOLOv8-main\archive (1)\Vehicle_Detection_Image_Dataset\sample_video.mp4: 384x640 3 vehicles, 4.8ms
video 1/1 (frame 3/601) d:\Self Projects\Real-Time-Vehicle-Detection-and-Counting-YOLOv8-main\ar

## 10. Conclusion

The YOLOv8s model was successfully fine-tuned for vehicle detection and tested on a traffic video.

The trained `best.pt` model is used in `counting.py` for vehicle tracking, entry and exit counting, traffic-level display, and annotated video generation.